# Zero shot and Few shot learning for text classification

In this notebook we'll explore the use of LangChain for **few shot learning** in the context of text classification.

**Few shot learning** is a form of machine learning where a model is taught to perform a particular task, such as classification or named entity recognition, by being provided with a few examples only, instead of a large training set, as is the case in 'traditional' machine learning tasks.

As any other task based on large languge models (LLMs), few shot learning relies on the use of appropriate prompts. LangChain offers prompt template classes specifically built for few shot learning, which we will use to classify a set of software requirements into functional and non-functional.

There is also **zero-shot learning**, where the model is asked to performe a specific task without any examples being given, only guidance / instruction in natural language (prompt). Still, it is often recommended to provide the model with at least one example.

For this class you'll need to install the following additional packages: `faiss-cpu`, `FlagEmbedding`, `langchain-huggingface`

[`faiss-cpu`](https://pypi.org/project/faiss-cpu/), is a python implentation of the Facebook's [FAISS](https://github.com/facebookresearch/faiss) library for efficient similarity search and clustering of text embeddings. The reason we need similarity search is that we want to select examples similar to the user's input (i.e., the instance to be classified).

`FlagEmbedding` and `langchain-huggingface` are libraries required for setting up an embeddings model we will use to create vector representation of examples for few-shot learning (such embeddings-based representation of examples is required for performing semantic search, as will be explained later).

If you haven't already done that for the previous class, you should also install: `langchain`, `langchain_community`, `langchain_groq`

In [ ]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

SEED = 9

### Instantiate an LLM chat model

As before, we will use Groq API to access `gpt-oss-20b` model. More precisely, we will leverage [LangChain's integration with Groq API](https://python.langchain.com/docs/integrations/chat/groq/), which further simplifies instantiation of LLMs that Groq offers.

To use Groq services, we need to load Groq API key into the current application environment (as shown below).

For more details about `gpt-oss-20b` model and access to it via Groq API, see [the LangChain Intro notebook](https://github.com/jeljov/NAP2026/blob/main/llms/LangChain_Intro_2026.ipynb).

In [ ]:
load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY is necessary for running an LLM in the code below. Please include it in the .env file")

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b",
               temperature=0.0,
               model_kwargs={'seed':SEED})

## Zero shot learning through prompting

Let's see an example where we will try to instruct an LLM to classify software requirements into functional and non-fuctional.

We start by checking if the model is 'aware' of these two main categories of software requirements:

In [ ]:
system_message = """
You are a well established software engineer with a lot of experience in gathering and managing software requirements.
You assist the user by answering their questions about software requirements in a clear and concise manner.
Restrict your answer to one paragraph only.
"""

user_message = "What are {req_type} requirements?"

initial_prompt = ChatPromptTemplate.from_messages([
    ('system', system_message),
    ('human', user_message),
])

simple_chain = initial_prompt | llm | StrOutputParser()

In [ ]:
resp_functional_req = simple_chain.invoke('functional')
print(resp_functional_req)

We will add a simple utility function that wraps the text so that it is easier to read the model's output:

In [ ]:
def wrap_and_print(text, width=100):
  import textwrap
  wrapped_text = textwrap.fill(text, width=width)
  print(wrapped_text +'\n')

In [ ]:
wrap_and_print(resp_functional_req)

In [ ]:
resp_nonfunctional_req = simple_chain.invoke('non-functional')
wrap_and_print(resp_nonfunctional_req)

Since the LLM is quite knowledgeable about software requirements, we may first try classification of requirements into functional and non-functional via **zero-shot learning**:

In [ ]:
zero_shot_system_msg = """
You are an experienced software engineer with expertise in requirements engineering.
Please help the user classify software requirements into functional and non-functional.
The user will give you a list of software requirements, each requirement starting in a new line.
Classify each requirement in the user's list into functional ("Functional") or non-functional ("Non-functional") categories.
Think through each requirement in the user's list before classifying it.
If you don't know how to classify a requirement, assign to it the "None" label.
Return your response as a JSON list of tuples, where the first element in the tuple is the text of a requirement from the user's list, while the second elements is the category assigned to that requirement (or "None" if you do no know how to classify it).
Your response should include only this JSON list and nothing else.
"""

zero_shot_user_msg="Software requirements to classify:\n{requirements}"

zero_shot_template = ChatPromptTemplate.from_messages([
    ('system', zero_shot_system_msg),
    ('human', zero_shot_user_msg),
])

zero_shot_simple_chain = zero_shot_template | llm | JsonOutputParser()

As an initial set of data for testing the zero-shot prompting, we'll use a small set of requirements in the form of user stories, for a (imaginary) music recommender mobile app:

In [ ]:
sample_requirements_list = [
    "I want to sign up with my email and password so I can access the app's features.",
    "I want to create, name, and modify a custom playlist so I can organize my favorite tracks.",
    "I want the music recommendation list to load in less than two seconds so I don't have to wait.",
    "I expect all my listening history and personal data to be securely encrypted so my information is protected.",
    "I want to be able to save a recommended song for offline playback so I can listen to it without an internet connection.",
    "I need the app to be available 99.9% of the time so I can reliably listen to music during my travels.",
    "I need the app's interface to look and behave consistently across all screens so it's intuitive and easy to use.",
    "I want to be able to search for music by typing an artist name, song title, or genre so I can quickly find what I'm looking for.",
    "I need the app to continue playing music if I briefly lose my cellular connection so my listening experience isn't interrupted.",
    "I want the app to use no more than 100MB of storage for cached data so my phone doesn't run out of space.",
    "I want to be able to 'Like' or 'Dislike' a recommended song so the system can improve future recommendations."
]

sample_requrements_str = "- " + "\n- ".join(sample_requirements_list)

In [ ]:
print(zero_shot_template.format(requirements = sample_requrements_str))

In [ ]:
zero_shot_classes = zero_shot_simple_chain.invoke(sample_requrements_str)

In [ ]:
type(zero_shot_classes)

In [ ]:
for i, classified_req in enumerate(zero_shot_classes):
  req, cls = classified_req
  print(f"{i+1}. {req}: {cls.upper()}")

This worked well, but as few-shot generally performs better than zero-shot and this was just a tiny sample, we'll next explore few-shot learning.

## Few shot learning

The idea behind **few shot learning** is to provide an LLM with some additional knowledge and / or instruction how to use the knowledge it has, to be able to perform a particular task.

Note: knowledge that an LLM 'develops' through training (both pre-training and instruction and preference tuning) is often referred to as **parametric knowledge**, whereas any additional knowledge and / or instruction it receives later, during the actual use is known as **contextual knowledge**.

In case of Chat LLMs, this *contextual knowledge*, comes in the form of a few pairs of interactions between the user and the chatbot, serving as examples of what we expect the model to do after receiving an input from the user. It has been shown that in large majority of cases, with only a few examples (3-4) and sometimes even only one example, a model can be well instructed for almost any specific task.

A nice quick intro to few shot learning (through prompting) is given in [this](https://youtu.be/JSBjj09xJeM?si=DJwLOeFHX7KOTGq2) short (5min) lecture from U. of Texas.

One of the main challenges in few-shot learning is the selection of examples to give to the model to learn from, as these should be sufficiently represenative of the user input. LangChain offers support for example selection, as we'll explore next.

### LangChain's support for few shot learning

To learn about dynamic few-shot prompting (i.e., few-shot prompting with dynamic example selection) , we will use a dataset of software requirements with human-assigned functional / non-functional labels. The dataset orginates from the "[Software requirements dataset](https://www.kaggle.com/datasets/iamvaibhav100/software-requirements-dataset)" available from Kaggle.com

We will start by loading the dataset (stored localy in the `software_requirements_processed.csv` file) and selecting a small sample of requirements to serve as examples for few shot learning, whereas the remaining ones will be used for evaluation purposes.

In [ ]:
import pandas as pd

req_data = pd.read_csv('data/software_requirements_processed.csv')
req_data.shape

In [ ]:
req_data.head(10)

We can examine the distribution of functional / non-functional requirements in this dataset:

In [ ]:
req_data.Category.value_counts(normalize=True)

Next, we take a small random sample - say 100 requirements - from which we will select (a much smaller subset of) examples for few shot learning

In [ ]:
req_ex_sample = req_data.sample(100, random_state=SEED)
req_ex_sample.Category.value_counts()

This random set of labelled requirements need to be turned into a list of examples, where each example is presented as a dictionary of two items: a requirement and its corresponding type (functional / non-functional), as exemplified below:

```
examples = [
    {
      "Requirement": "This is a placeholder for a real software requirement",
      "Category": "Functional"
    },
    {
      "Requirement": "This is a placeholder for a real software requirement",
      "Category": "Non-functional"
    },
    #....
]
```
To create such a list easily, we change the name of the data frame columns and transform the data frame into a list of dictionaries:

In [ ]:
req_ex_list = req_ex_sample[['Requirement','Category']].to_dict('records')
display(len(req_ex_list))
display(req_ex_list[:5])

To select a subset of examples to be used for few shot classification of a user's input (requirement), we will use LangChain's `SemanticSimilarityExampleSelector`, an example selector that selects examples so that they are highly similar to the user input.

To compute similarity between the user's input and the examples, we need to represent the inputs and the examples in a numerical format, suitable for (similarity) computation. To that end, we will use **text embeddings** as currently the most widely used approach for representing textual content.  

Specifically, we will use the [bge-large-en-v1.5](https://huggingface.co/BAAI/bge-large-en-v1.5) open source model for creating text embeddings. It was selected due to its small size and good performance. It was developed by the Beijing Academy of AI and as other open source language models, it is available from the [HuggingFace Hub](https://huggingface.co/models).

You may want to explore the [embedding models leaderboard at HuggingFace](https://huggingface.co/spaces/mteb/leaderboard) to find alternative models you may want to try out.

Note that LangChain directly supports large number of embedding models; for a list of supported models and instructions (code examples) how to use them, go to [Integrations > Embedding models](https://docs.langchain.com/oss/python/integrations/embeddings) in LangChain documentation.

A reminder: To access a model hosted at HuggingFace Hub, you'll need a HuggingFace (HF) access token. To obtain such a token, you should first create an account at [HuggingFace.co](https://huggingface.co/). Then, click on your profile in the top-right corner, then *Settings*, then *Access Tokens*, then *New Token*, set *Role* to *write* and *Generate*.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

if not os.getenv("HF_TOKEN"):
    print("HF_TOKEN is necessary for pulling a model from HuggingFace. Please include it in the .env file")

model_name = "BAAI/bge-large-en-v1.5"

bge_embeddings = HuggingFaceEmbeddings(
  model_name=model_name,
  model_kwargs={'device': 'cpu'}, # tells it to use CPU; alternative is 'cuda' to run on GPU
  encode_kwargs={'normalize_embeddings': True} # normalising embeddings as that is required for the computation of cosine similarity (which will be used later)
)

Next, we instantiate the `SemanticSimilarityExampleSelector`, with the BGE embeddings model and [FAISS](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/) as the vectors store.

FAISS is an in-memory vector store, which allows for quick search of (embedded / vectorised) documents based on different similarity measures

In [ ]:
from langchain_core.example_selectors.semantic_similarity import SemanticSimilarityExampleSelector
from langchain_community.vectorstores.faiss import FAISS

example_selector = SemanticSimilarityExampleSelector.from_examples(
    # the list of examples available to select from
    examples=req_ex_list,
    # the embeddings model to be used to represent the examples, so that their semantic similarity can be computed
    embeddings=bge_embeddings,
    # The VectorStore class that is used to store the embeddings and do a similarity search over
    vectorstore_cls=FAISS,
    # The number of examples to select
    k=3
)

We can first try it out by taking one requirement from the part of the data set that was not used for buidling the example vector store:

In [ ]:
example_out_of_sample = req_data.drop(req_ex_sample.index.tolist()).Requirement.sample(random_state=SEED)
print(example_out_of_sample.iloc[0])

In [ ]:
sample_input = {"Requirement": example_out_of_sample.iloc[0]}

selected_examples = example_selector.select_examples(sample_input)

for ex in selected_examples:
  print(f"Requirement: \"{ex['Requirement']}\": {ex['Category']}")

Now we can create a few shot prompt with examples selected by the example selector. To that end, we will format the examples so that they can be directly added to a prompt:

In [ ]:
selected_examples_str = ""
for ex in selected_examples:
  selected_examples_str += f"Human: {ex['Requirement']}\nAI: {ex['Category']}\n"
print(selected_examples_str)

In [ ]:
few_shot_system_msg = """
You are an experienced software engineer with expertise in requirements engineering. Please help the user classify software requirements into functional and non-functional.
Think through the requirement the user gives you before responding.
Limit your response to one term: "Functional" or "Non-functional".
If you don't know the answer, respond with "None".

Below are a couple of examples of classified requirements, to help you with the classification task.
{examples}
"""

few_shot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", few_shot_system_msg),
        ("user", "{input}"),
    ]
)

In [ ]:
few_shot_chain = few_shot_prompt | llm

In [ ]:
test_req_resp = few_shot_chain.invoke({'examples':selected_examples_str, 'input':sample_input['Requirement']})
print(test_req_resp.content)

## Evaluation with human labelled data

Let's now take the remaining part of the dataset we've loaded to examine how well the model performs the classification task. To that end, we will first drop instances that were used for examples selection (to avoid 'data leakage'):

In [ ]:
example_indices = req_ex_sample.index.tolist()
req_data.drop(example_indices, inplace=True)

In [ ]:
req_data.shape

For evaluation purposes, we will take a random sample of requirements that the model has not seen. We are taking a small sample, in order not to overstep the available free time / tokens on Groq.

In [ ]:
eval_sample_size = 50

req_eval_sample = req_data.sample(eval_sample_size, random_state=SEED)
req_eval_sample.reset_index(drop=True, inplace=True)
req_eval_sample.head()

Now, we can pass the test requirements to the previously built `few_shot_chain` and add a new column to the data frame based on the obtained results. Along the way, we will also compute the number of tokens used to get a better idea of how many tokens we consume.

In [ ]:
def add_tokens_used(tokens_used, groq_response):
  token_metadata = groq_response.response_metadata['token_usage']

  tokens_used['prompt_tokens'] += token_metadata['prompt_tokens']
  tokens_used['completion_tokens'] += token_metadata['completion_tokens']
  tokens_used['reasoning_tokens'] += token_metadata['completion_tokens_details']['reasoning_tokens']
  tokens_used['total_tokens'] += token_metadata['total_tokens']

In [ ]:
def select_and_prepare_examples(requirement_text):
  selection = example_selector.select_examples({'Requirement':requirement_text})
  selection_str = ""
  for ex in selection:
    selection_str += f"Human: {ex['Requirement']}\nAI: {ex['Category']}\n"
  return selected_examples_str

In [ ]:
from time import sleep
from collections import defaultdict

results = []
errors = []

token_usage = defaultdict(int)

for i, req in enumerate(req_eval_sample.Requirement.to_list()):

  if i % 10 == 0: print(f"processing {i+1}. requirement")

  try:
    llm_response = few_shot_chain.invoke({
        'examples':select_and_prepare_examples(req),
        'input':req
        })

    add_tokens_used(token_usage, llm_response)

    if llm_response.content.lower() in ['functional', 'non-functional']:
      results.append({
        'Requirement': req,
        'Predicted_category': llm_response.content
      })
    else:
      print(f"Incorrect label ({llm_response.content}) for requirement '{req}'")
      errors.append({
          'Requirement': req,
          'Predicted_category': llm_response.content
      })

  except Exception as e:
    print(f"The following error occurred when classifying requirement \"{req}\": {e}")

  # put the model to sleep for 3 sec to avoid running into the Groq's rate limit
  sleep(3)

results_df = pd.DataFrame(results)

In [ ]:
results_df.head()

In [ ]:
req_eval_sample = pd.merge(req_eval_sample, results_df, on='Requirement', how='inner')

In [ ]:
req_eval_sample[['Requirement', 'Category', 'Predicted_category']].head()

Check the amount of tokens used:

In [ ]:
from pprint import pprint

pprint(dict(token_usage))

We are well within the Groq's rate limits: https://console.groq.com/docs/rate-limits

Check the level of matching between human labels and the labels assigned by the LLM:

In [ ]:
from statistics import mean

def label_match(row):
  human_lbl = row['Category']
  llm_lbl = row['Predicted_category']
  return human_lbl == llm_lbl

req_type_match = req_eval_sample.apply(label_match, axis=1)
print(f"Number of matched labels: {sum(req_type_match)}")
print(f"Proportion of matched labels: {mean(req_type_match):.4f}")

We can also compute the reqular classification eval metrics (confusion matrix, precision, recall, F1):

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_true=req_eval_sample.Category,
                                        y_pred=req_eval_sample.Predicted_category,
                                        cmap='Blues');

In [ ]:
print(classification_report(y_true=req_eval_sample.Category, y_pred=req_eval_sample.Predicted_category))

Let's examine where the LLM made mistakes:

In [ ]:
for error in errors:
  print(error['Requirement'])
  print(error['Predicted_category'])
  print('-'*70)

In [ ]:
req_eval_sample['correct_cls'] = req_type_match
req_eval_sample.loc[~req_eval_sample.correct_cls, ['Requirement','Category','Predicted_category']]

Save the results in a csv file:

In [ ]:
fname = "sample_requirements_labelled.csv"
req_eval_sample.to_csv(fname, index=False)